# Capa Bronze — Ingesta de datos crudos

Primer paso de la arquitectura Medallion: leer los CSV desde Azure Blob Storage y persistirlos en formato Delta sin ninguna transformación.

In [0]:
# Leés el valor que mandó Data Factory:
storage_key = dbutils.widgets.get("storage_key")

# Validás que no venga vacío
if not storage_key:
    raise ValueError("El parámetro storage_key no fue provisto.")

## Paso 1 — Configurar acceso al Storage Account

In [0]:
# Acceso directo al storage (sin mount, compatible con clusters Serverless)
storage_account = "icarostorage" # Reemplazar con tu storage
container       = "tpf-medallion-data" #Crear contenedor para medallion
key             = storage_key

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    key
)

BASE = f"wasbs://{container}@{storage_account}.blob.core.windows.net"

# Verificar que los archivos son accesibles
dbutils.fs.ls(BASE)


## Paso 2 — Leer los CSV y escribir en Delta (Bronze)

Leemos cada archivo con `header=True` y todos los campos quedan como `string`. Esto es intencional en Bronze: minimizamos el riesgo de rechazar datos por tipos incorrectos.

In [0]:
from pyspark.sql import functions as F

# 1. Leer el JSON crudo de la API con multiline activo
df_api_raw = spark.read.option("multiline", "true").json(f"{BASE}/pokemon.json")

# 2. Guardar en Bronze tal cual vino (formato Delta)
df_api_raw.write.format("delta").mode("overwrite").save(f"{BASE}/pokemon")

print("JSON Bronze guardado con éxito.")

In [0]:
# Leer y guardar las tablas maestras en Bronze
pokemones_bronze  = spark.read.csv(f"{BASE}/pokemon_metadata.csv",           header=True)

pokemones_bronze.write.format("delta").mode("overwrite").save(f"{BASE}/pokemones_metadata")

print("clientes:",  pokemones_bronze.count(),  "filas")

## Paso 3 — Verificación

Comprobamos que las tablas Delta se escribieron correctamente.

In [0]:
# Verificar Bronze
print("=== ventas_diarias ===")
spark.read.format("delta").load(f"{BASE}/pokemones_metadata").printSchema()
spark.read.format("delta").load(f"{BASE}/pokemones_metadata").show(5)

In [0]:
# Verificar Bronze
print("=== ventas_diarias ===")
spark.read.format("delta").load(f"{BASE}/pokemon").printSchema()
spark.read.format("delta").load(f"{BASE}/pokemon").show(5)